In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PATH DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

CHECKPOINT_DIR=os.path.join(
    BASE_DIR,
    "checkpoints"
)
print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

In [ ]:
#install image bind
!git clone https://github.com/facebookresearch/ImageBind.git
%cd ImageBind

!pip install -r requirements.txt
!pip install .

In [ ]:
##download pretrained weights
!mkdir checkpoints

!wget https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth \
-O checkpoints/imagebind_huge.pth

In [ ]:
#imports
import torch
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

In [ ]:
#load model
device = "cuda"

model = imagebind_model.imagebind_huge(
    pretrained=True
)

model.eval()

model.to(device)

In [ ]:
#creating output folders
import os

IB_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "imagebind_image_embeddings"
)

IB_AUDIO_DIR = os.path.join(
    BASE_DIR,
    "imagebind_audio_embeddings"
)

IB_VIDEO_DIR = os.path.join(
    BASE_DIR,
    "imagebind_video_embeddings"
)

os.makedirs(
    IB_IMAGE_DIR,
    exist_ok=True
)

os.makedirs(
    IB_AUDIO_DIR,
    exist_ok=True
)

os.makedirs(
    IB_VIDEO_DIR,
    exist_ok=True
)


In [ ]:
#common ids
import pandas as pd
import os

common_ids = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "common_samples.csv"
    )
)["video_id"].tolist()

print("Total videos:", len(common_ids))
print(common_ids[:5])

In [ ]:
#generate image embeddings
from tqdm import tqdm
import numpy as np

for vid in tqdm(common_ids):

    image_path = os.path.join(
        FRAME_DIR,
        vid + ".jpg"
    )

    inputs = {

        ModalityType.VISION:
        data.load_and_transform_vision_data(
            [image_path],
            device
        )
    }

    with torch.no_grad():

        emb = model(inputs)[
            ModalityType.VISION
        ]

    np.save(

        os.path.join(
            IB_IMAGE_DIR,
            vid + ".npy"
        ),

        emb.cpu().numpy().squeeze()
    )

In [ ]:
#generating audio embeddings
for vid in tqdm(common_ids):

    audio_path = os.path.join(
        AUDIO_DIR,
        vid + ".wav"
    )

    inputs = {

        ModalityType.AUDIO:
        data.load_and_transform_audio_data(
            [audio_path],
            device
        )
    }

    with torch.no_grad():

        emb = model(inputs)[
            ModalityType.AUDIO
        ]

    np.save(

        os.path.join(
            IB_AUDIO_DIR,
            vid + ".npy"
        ),

        emb.cpu().numpy().squeeze()
    )

In [ ]:
!pip install decord

In [ ]:
#generating video embeddings
from tqdm.auto import tqdm
import numpy as np
for vid in tqdm(common_ids):

    video_path = os.path.join(
        VIDEO_DIR,
        vid + ".mp4"
    )

    inputs = {

        ModalityType.VISION:
        data.load_and_transform_video_data(
            [video_path],
            device
        )
    }

    with torch.no_grad():

        emb = model(inputs)[
            ModalityType.VISION
        ]

    np.save(

        os.path.join(
            IB_VIDEO_DIR,
            vid + ".npy"
        ),

        emb.cpu().numpy().squeeze()
    )

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

common_ids = pd.read_csv(
    f"{OUTPUT_DIR}/common_samples.csv"
)["video_id"].tolist()

train_ids, temp_ids = train_test_split(
    common_ids,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

print(len(test_ids))

In [ ]:
import numpy as np
import os

image_embs = []
audio_embs = []
video_embs = []

for vid in test_ids:

    image_embs.append(
        np.load(
            os.path.join(
                IB_IMAGE_DIR,
                vid + ".npy"
            )
        )
    )

    audio_embs.append(
        np.load(
            os.path.join(
                IB_AUDIO_DIR,
                vid + ".npy"
            )
        )
    )

    video_embs.append(
        np.load(
            os.path.join(
                IB_VIDEO_DIR,
                vid + ".npy"
            )
        )
    )

image_embs = np.array(image_embs)
audio_embs = np.array(audio_embs)
video_embs = np.array(video_embs)

print(image_embs.shape)
print(audio_embs.shape)
print(video_embs.shape)

In [ ]:
#zero shot fusion
from sklearn.preprocessing import normalize

fused = (
    image_embs
    +
    audio_embs
) / 2
#normalise
fused = normalize(fused)
video_embs = normalize(video_embs)

In [ ]:
#cosine simlarity
similarity_matrix = fused @ video_embs.T

diag_sim = np.mean(
    np.diag(
        similarity_matrix
    )
)

print(
    "Cosine Similarity:",
    diag_sim
)

In [ ]:
#reterival metrics
ranks = []

for i in range(len(fused)):

    sims = similarity_matrix[i]

    sorted_idx = np.argsort(
        sims
    )[::-1]

    rank = np.where(
        sorted_idx == i
    )[0][0] + 1

    ranks.append(rank)

r1 = np.mean(
    np.array(ranks) <= 1
)

r5 = np.mean(
    np.array(ranks) <= 5
)

r10 = np.mean(
    np.array(ranks) <= 10
)

medr = np.median(
    ranks
)

print(
    f"Recall@1 : {r1*100:.2f}%"
)

print(
    f"Recall@5 : {r5*100:.2f}%"
)

print(
    f"Recall@10 : {r10*100:.2f}%"
)

print(
    f"Median Rank : {medr}"
)

In [ ]:
#mse
ranks = []

for i in range(len(fused)):

    sims = similarity_matrix[i]

    sorted_idx = np.argsort(
        sims
    )[::-1]

    rank = np.where(
        sorted_idx == i
    )[0][0] + 1

    ranks.append(rank)

r1 = np.mean(
    np.array(ranks) <= 1
)

r5 = np.mean(
    np.array(ranks) <= 5
)

r10 = np.mean(
    np.array(ranks) <= 10
)

medr = np.median(
    ranks
)

print(
    f"Recall@1 : {r1*100:.2f}%"
)

print(
    f"Recall@5 : {r5*100:.2f}%"
)

print(
    f"Recall@10 : {r10*100:.2f}%"
)

print(
    f"Median Rank : {medr}"
)

In [ ]:
#parameter count
import torch

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
#inference time
#for video embeddings
import time
import torch

model.eval()

starter = time.time()

with torch.no_grad():

    emb = model(inputs)

ender = time.time()

latency = (
    ender - starter
) * 1000

print(
    f"Inference Latency: {latency:.2f} ms"
)

In [ ]:
#average
import time
import numpy as np

times = []

model.eval()

for _ in range(50):

    start = time.time()

    with torch.no_grad():

        _ = model(inputs)

    end = time.time()

    times.append(
        (end - start) * 1000
    )

print(
    "Average Latency:",
    np.mean(times),
    "ms"
)

print(
    "Std:",
    np.std(times),
    "ms"
)